# Draw profiles on Amy's pilot data with exact genmag 

20260916
Kyoko Kusano

In [6]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time

import matplotlib.pyplot as plt

from qstr_dataset import datasets, paths
from qstr_dataset.profile_builder import magnitude_profile, magnitude_profiles
from qstr_dataset.profile_plot import plot_magnitude_profile


In [ ]:
SUB_NO = 0
dissim = datasets.dissimilarity("amy", sub_no=SUB_NO)

start = time.time()
result = magnitude_profile(dissim)
axes, trust_axes = plot_magnitude_profile(
    result, title=f"Amy sub {SUB_NO}", ylim=(0, 12))

print(f"elapsed      : {time.time() - start:.0f} s")
print(f"limit        : {result.limit}   ({result.limit_decimal})")
print(f"  route      : {result.limit_method}, "
      f"{'proved' if result.limit_is_proved else 'certified digits'}")
print(f"m(A(inf))    : {result.limit_at_infinity}   <- where double precision lands")
print(f"rank at inf  : {result.rank_at_infinity} / {result.size}")
print(f"curve route  : {result.curve_method}")
print(f"poles ({len(result.poles)})    : {[round(pole, 6) for pole in result.poles]} "
      f"({result.poles_method})")
for note in result.notes:
    print(f"  note: {note}")


In [ ]:
# All 120 subjects. The symbolic route finishes for most of them, so this takes
# minutes rather than seconds; every finished subject is cached under
# data/interim/magnitude_profiles, so a second run returns immediately and an
# interrupted one resumes.

dissims = datasets.amy_all_subjects()

summary, results = magnitude_profiles(
    dissims,
    name="amy",
    max_workers=max(1, (os.cpu_count() or 2) - 1),
)
summary.to_csv(paths.table("amy_magnitude_profiles.csv"))
print(summary["status"].value_counts())
print(summary["limit_method"].value_counts())
summary.head()


In [ ]:
# Draw a few of them. Each call makes its own figure, with the panel showing
# how far double precision can be trusted underneath.

targets = [0, 1, 2, 3]
for sub_no in targets:
    if sub_no not in results:
        print(f"sub {sub_no}: {summary.loc[sub_no, 'status']}")
        continue
    plot_magnitude_profile(results[sub_no], title=f"Amy sub {sub_no}", ylim=(-1, 24))
